# DB MoveOptimizer - Analyst Agent Foundation

Building on the API connection established in notebook 01, this notebook introduces the first meaningful agent behavior:
a **DB Mobility Analyst** persona that receives a mock user travel record and returns a structured pattern summary.

The graph topology stays identical (no new nodes), but we add:
- A **system prompt** that gives the model its role as a Deutsche Bahn mobility analyst
- **Mock travel data** for one synthetic user (5 trips)
- Updated  that formats the data into the API call

## Notebook Outline
1. Environment Setup and API Configuration
2. Initialize OpenAI Client with University Endpoint
3. Define Mock Travel Data
4. Create Chat Function with System Prompt Support
5. Set Up LangGraph State and Nodes (updated)
6. Build LangGraph Workflow
7. Test: Analyst Agent on Mock User

## 1. Environment Setup and API Configuration

In [1]:
import subprocess
import sys

def install_packages(packages):
    for package in packages:
        try:
            __import__(package.split("==")[0].replace("-", "_"))
            print(f"✓ {package} already installed")
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
            print(f"✓ {package} installed")

install_packages(["openai", "langgraph", "langchain", "langchain-core", "langchain-openai"])

✓ openai already installed
✓ langgraph already installed
✓ langchain already installed
✓ langchain-core already installed
✓ langchain-openai already installed


In [2]:
import os

UNI_GPT_BASE_URL = "https://chat.kiconnect.nrw/api/v1"
UNI_GPT_MODEL    = "Openai GPT OSS 120B"
UNI_GPT_API_KEY  = os.getenv("UNI_GPT_API_KEY", "")

if not UNI_GPT_API_KEY:
    from getpass import getpass
    UNI_GPT_API_KEY = getpass("Enter your University GPT API Key: ")

print("=" * 60)
print("University GPT API Configuration")
print("=" * 60)
print(f"Base URL: {UNI_GPT_BASE_URL}")
print(f"Model:    {UNI_GPT_MODEL}")
print(f"API Key:  {'***' + UNI_GPT_API_KEY[-8:] if UNI_GPT_API_KEY else 'NOT SET'}")
print("=" * 60)

University GPT API Configuration
Base URL: https://chat.kiconnect.nrw/api/v1
Model:    Openai GPT OSS 120B
API Key:  ***jQjSDag=


## 2. Initialize OpenAI Client with University Endpoint

In [3]:
from openai import OpenAI

client = OpenAI(base_url=UNI_GPT_BASE_URL, api_key=UNI_GPT_API_KEY)
print("✓ OpenAI client initialized with University endpoint")

✓ OpenAI client initialized with University endpoint


## 3. Define Mock Travel Data

One synthetic user with 5 recent trips. This simulates what a real DB Navigator export would look like.
Fields match the schema outlined in .

In [4]:
import json

# Synthetic user profile
MOCK_USER = {
    "user_id": "USR-0042",
    "name": "Anna Müller",
    "current_subscriptions": ["BahnCard 25 (2nd class)"],
    "travel_log": [
        {"date": "2026-05-02", "origin": "Köln Hbf",    "destination": "Frankfurt Hbf",  "ticket_type": "Flexpreis",     "price_eur": 89.00, "class": 2},
        {"date": "2026-05-09", "origin": "Frankfurt Hbf", "destination": "Köln Hbf",    "ticket_type": "Sparpreis",     "price_eur": 17.90, "class": 2},
        {"date": "2026-05-16", "origin": "Köln Hbf",    "destination": "Frankfurt Hbf",  "ticket_type": "Flexpreis",     "price_eur": 89.00, "class": 2},
        {"date": "2026-05-23", "origin": "Frankfurt Hbf", "destination": "Düsseldorf Hbf", "ticket_type": "Deutschlandticket", "price_eur": 49.00, "class": 2},
        {"date": "2026-05-30", "origin": "Köln Hbf",    "destination": "Frankfurt Hbf",  "ticket_type": "Sparpreis",     "price_eur": 24.90, "class": 2},
    ]
}

print(f"✓ Mock user loaded: {MOCK_USER['name']} ({MOCK_USER['user_id']})")
print(f"  Subscriptions: {MOCK_USER['current_subscriptions']}")
print(f"  Trips in log:  {len(MOCK_USER['travel_log'])}")

✓ Mock user loaded: Anna Müller (USR-0042)
  Subscriptions: ['BahnCard 25 (2nd class)']
  Trips in log:  5


## 4. Create Chat Function with System Prompt Support

The only change from notebook 01: the function now accepts an optional  argument
that is prepended as a  message. Everything else stays the same.

In [5]:
DB_ANALYST_SYSTEM_PROMPT = """You are a Deutsche Bahn mobility analyst.
Your job is to review a single user's recent travel log and produce a concise pattern summary.

Always respond with:
1. DOMINANT ROUTE — the most frequent origin-destination pair
2. TICKET MIX — breakdown of ticket types used
3. SPEND SUMMARY — total spend and average cost per trip
4. ONE OBSERVATION — a single sentence noting an inefficiency or opportunity

Be factual and brief. Do not make recommendations yet — that is handled by a separate agent."""


def call_university_gpt(messages, system_prompt=None, max_new_tokens=512, temperature=0.0):
    """
    Call the University of Cologne GPT endpoint.
    Prepends system_prompt as a system message when provided.
    """
    full_messages = []
    if system_prompt:
        full_messages.append({"role": "system", "content": system_prompt})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model=UNI_GPT_MODEL,
        messages=full_messages,
        max_tokens=max_new_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()


print("✓ call_university_gpt updated with system_prompt support")
print("✓ DB_ANALYST_SYSTEM_PROMPT defined")

✓ call_university_gpt updated with system_prompt support
✓ DB_ANALYST_SYSTEM_PROMPT defined


## 5. Set Up LangGraph State and Nodes

State now carries a  field. The  node serialises it into
the user message so the model always receives structured input.

In [8]:
from typing import TypedDict, List, Any, Optional
from langgraph.graph import StateGraph, START, END


class AnalystState(TypedDict):
    """State for the Analyst agent workflow."""
    travel_data: dict           # Raw user travel record
    messages: List[dict]        # Assembled messages sent to the API
    response: str               # Model output
    status: str                 # processing | complete | error


def input_processor(state: AnalystState) -> AnalystState:
    """Format travel_data into a user message for the analyst prompt."""
    user_id   = state["travel_data"].get("user_id", "unknown")
    name      = state["travel_data"].get("name", "unknown")
    subs      = ", ".join(state["travel_data"].get("current_subscriptions", []))
    trips_json = json.dumps(state["travel_data"].get("travel_log", []), indent=2)

    user_message = (
        f"Analyse the following travel log for user {name} ({user_id})."
        f"Current subscriptions: {subs}"
        f"Travel log (JSON):{trips_json}"
    )

    state["messages"] = [{"role": "user", "content": user_message}]
    state["status"]   = "processing"
    print(f"✓ input_processor: formatted {len(state['travel_data'].get('travel_log',[]))} trips for {name}")
    return state


def api_caller(state: AnalystState) -> AnalystState:
    """Call the University GPT API with the DB analyst system prompt."""
    try:
        response = call_university_gpt(
            messages=state["messages"],
            system_prompt=DB_ANALYST_SYSTEM_PROMPT,
            max_new_tokens=512,
            temperature=0.0,
        )
        state["response"] = response
        state["status"]   = "complete"
        print("✓ api_caller: response received")
    except Exception as e:
        state["response"] = f"Error: {e}"
        state["status"]   = "error"
        print(f"✗ api_caller failed: {e}")
    return state


def output_formatter(state: AnalystState) -> AnalystState:
    """Pass-through formatter — prints the analysis to stdout."""
    if state["status"] == "complete":
        print("" + "=" * 60)
        print("ANALYST OUTPUT")
        print("=" * 60)
        print(state["response"])
        print("=" * 60)
    return state


print("✓ AnalystState and node functions defined")

✓ AnalystState and node functions defined


## 6. Build LangGraph Workflow

Same linear topology as notebook 01.

In [9]:
workflow = StateGraph(AnalystState)

workflow.add_node("input_processor", input_processor)
workflow.add_node("api_caller",       api_caller)
workflow.add_node("output_formatter", output_formatter)

workflow.add_edge(START,              "input_processor")
workflow.add_edge("input_processor",  "api_caller")
workflow.add_edge("api_caller",       "output_formatter")
workflow.add_edge("output_formatter", END)

graph = workflow.compile()

print("✓ Analyst workflow compiled")
print("""
Workflow:
  START
    ↓
  input_processor   ← formats travel_data into user message
    ↓
  api_caller        ← calls GPT with DB analyst system prompt
    ↓
  output_formatter  ← prints structured analysis
    ↓
  END
""")

✓ Analyst workflow compiled

Workflow:
  START
    ↓
  input_processor   ← formats travel_data into user message
    ↓
  api_caller        ← calls GPT with DB analyst system prompt
    ↓
  output_formatter  ← prints structured analysis
    ↓
  END



## 7. Test: Analyst Agent on Mock User

Run the full workflow with Anna Müller's travel log.

In [11]:
initial_state = {
    "travel_data": MOCK_USER,
    "messages":    [],
    "response":    "",
    "status":      "pending",
}

result = graph.invoke(initial_state)

print(f"Final status: {result['status']}")

✓ input_processor: formatted 5 trips for Anna Müller
✓ api_caller: response received
ANALYST OUTPUT
**DOMINANT ROUTE** – Köln Hbf → Frankfurt Hbf (3 trips)  

**TICKET MIX** – Flexpreis: 2 × (40 %) Sparpreis: 2 × (40 %) Deutschlandticket: 1 × (20 %)  

**SPEND SUMMARY** – Total €269.80; average per trip ≈ €53.96  

**ONE OBSERVATION** – The repeated Flexpreis tickets for Köln‑Frankfurt cost far more than the available Sparpreis or the flat‑rate Deutschlandticket could have covered.
Final status: complete
